In [0]:
silver_table = "asadproject.schemaproject.silver_weather"

df_silver = spark.table(silver_table)

display(df_silver)
print("rows:", df_silver.count())

time,temperature_c,wind_speed_kmh,precipitation_mm,humidity_pct,apparent_temperature_c,cloud_cover_pct,visibility_m,soil_temperature_c,weather_date,hour_of_day,ingestionTime
2026-10-30T01:00:00.000Z,23.3,1.8,0.0,95,28.1,5,5520.0,22.5,2026-10-30,1,2026-11-08T04:51:23.176Z
2026-10-30T16:00:00.000Z,29.0,9.1,0.0,61,31.9,19,10060.0,28.6,2026-10-30,16,2026-11-08T04:51:23.176Z
2026-10-31T04:00:00.000Z,22.1,1.9,0.0,95,26.3,6,5520.0,21.5,2026-10-31,4,2026-11-08T04:51:23.176Z
2026-10-31T19:00:00.000Z,26.4,4.9,0.0,77,30.5,36,8960.0,25.1,2026-10-31,19,2026-11-08T04:51:23.176Z
2026-11-01T11:00:00.000Z,30.5,4.2,0.0,57,35.4,43,11020.0,35.0,2026-11-01,11,2026-11-08T04:51:23.176Z
2026-11-02T05:00:00.000Z,22.7,5.5,0.0,89,26.2,8,6680.0,22.2,2026-11-02,5,2026-11-08T04:51:23.176Z
2026-11-02T14:00:00.000Z,32.3,7.3,0.0,50,35.5,95,12620.0,34.5,2026-11-02,14,2026-11-08T04:51:23.176Z
2026-11-03T06:00:00.000Z,23.5,4.0,0.0,97,28.3,1,4640.0,23.8,2026-11-03,6,2026-11-08T04:51:23.176Z
2026-11-03T12:00:00.000Z,31.6,6.7,0.1,59,37.0,100,8720.0,35.3,2026-11-03,12,2026-11-08T04:51:23.176Z
2026-11-04T14:00:00.000Z,31.5,5.5,0.0,57,35.5,100,10900.0,33.8,2026-11-04,14,2026-11-08T04:51:23.176Z


rows: 168


In [0]:
from pyspark.sql import functions as F

df_daily = (df_silver
    .groupBy("weather_date")
    .agg(
        F.round(F.avg("temperature_c"), 1).alias("avg_temp_c"),
        F.max("temperature_c").alias("max_temp_c"),
        F.min("temperature_c").alias("min_temp_c"),
        F.round(F.sum("precipitation_mm"), 1).alias("total_rain_mm"),
        F.max("wind_speed_kmh").alias("max_wind_kmh"),
        F.round(F.avg("humidity_pct"), 1).alias("avg_humidity_pct"),
        F.count("*").alias("hours_recorded"))
    .orderBy("weather_date"))

display(df_daily)

weather_date,avg_temp_c,max_temp_c,min_temp_c,total_rain_mm,max_wind_kmh,avg_humidity_pct,hours_recorded
2026-10-30,25.7,29.9,22.5,1.7,9.4,80.2,24
2026-10-31,26.4,31.4,21.9,0.0,8.3,75.1,24
2026-11-01,26.6,31.8,22.3,0.0,9.6,71.8,24
2026-11-02,26.5,32.3,22.6,2.9,14.0,76.1,24
2026-11-03,25.9,32.1,22.8,3.3,6.7,86.8,24
2026-11-04,25.5,31.8,23.0,4.8,12.7,87.3,24
2026-11-05,25.1,28.9,21.8,3.0,5.5,86.2,24


In [0]:
from pyspark.sql import functions as F

df_gold = (df_daily
    .withColumn("heat_risk",
        F.when(F.col("max_temp_c") >= 40, "High")
         .when(F.col("max_temp_c") >= 35, "Medium")
         .otherwise("Low"))
    .withColumn("rain_risk",
        F.when(F.col("total_rain_mm") >= 10, "High")
         .when(F.col("total_rain_mm") >= 2, "Medium")
         .otherwise("Low"))
    .withColumn("wind_risk",
        F.when(F.col("max_wind_kmh") >= 40, "High")
         .when(F.col("max_wind_kmh") >= 25, "Medium")
         .otherwise("Low")))

display(df_gold)

weather_date,avg_temp_c,max_temp_c,min_temp_c,total_rain_mm,max_wind_kmh,avg_humidity_pct,hours_recorded,heat_risk,rain_risk,wind_risk
2026-10-30,25.7,29.9,22.5,1.7,9.4,80.2,24,Low,Low,Low
2026-10-31,26.4,31.4,21.9,0.0,8.3,75.1,24,Low,Low,Low
2026-11-01,26.6,31.8,22.3,0.0,9.6,71.8,24,Low,Low,Low
2026-11-02,26.5,32.3,22.6,2.9,14.0,76.1,24,Low,Medium,Low
2026-11-03,25.9,32.1,22.8,3.3,6.7,86.8,24,Low,Medium,Low
2026-11-04,25.5,31.8,23.0,4.8,12.7,87.3,24,Low,Medium,Low
2026-11-05,25.1,28.9,21.8,3.0,5.5,86.2,24,Low,Medium,Low


In [0]:
gold_table = "asadproject.schemaproject.gold_weather_daily"
gold_path = "abfss://gold@strgproject.dfs.core.windows.net/weather_daily"

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_path)
    .saveAsTable(gold_table))
    

In [0]:
gold_table = "asadproject.schemaproject.gold_weather_daily"
gold_path = "abfss://gold@strgproject.dfs.core.windows.net/weather_daily"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table}
    USING DELTA
    LOCATION '{gold_path}'
""")

display(spark.table(gold_table))

weather_date,avg_temp_c,max_temp_c,min_temp_c,total_rain_mm,max_wind_kmh,avg_humidity_pct,hours_recorded,heat_risk,rain_risk,wind_risk
2026-10-30,25.7,29.9,22.5,1.7,9.4,80.2,24,Low,Low,Low
2026-10-31,26.4,31.4,21.9,0.0,8.3,75.1,24,Low,Low,Low
2026-11-01,26.6,31.8,22.3,0.0,9.6,71.8,24,Low,Low,Low
2026-11-02,26.5,32.3,22.6,2.9,14.0,76.1,24,Low,Medium,Low
2026-11-03,25.9,32.1,22.8,3.3,6.7,86.8,24,Low,Medium,Low
2026-11-04,25.5,31.8,23.0,4.8,12.7,87.3,24,Low,Medium,Low
2026-11-05,25.1,28.9,21.8,3.0,5.5,86.2,24,Low,Medium,Low
